# Last.fm bench

Compare embedding recommendations against Last.fm results.

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

## Data Prep

In [ ]:
DATA_DIR = Path("../datasets")
BENCH_PATH = DATA_DIR / "lastfm_recommendations_all_top_listener.csv"
RECS_PATH = DATA_DIR / "recommendations_album_level_avg_embeddings.parquet"
CATALOG_PATH = DATA_DIR / "albums.csv"

# Number of top recommendations to keep for each query album
TOP_N = 4

In [ ]:
# utility functions
def _top_n_subset(df, top_n):
    """Keep ranks 1..top_n and drop query albums with fewer than top_n recs."""
    sub = df[df["rank"] <= top_n].copy()
    keys = ["query_artist", "query_album"]
    n_recs = sub.groupby(keys, observed=True)["rank"].transform("count")
    return sub[n_recs >= top_n].reset_index(drop=True)


def keep_top_n(df, top_n=TOP_N, verbose=True):
    keys = ["query_artist", "query_album"]
    n_queries_before = df[df["rank"] <= top_n].groupby(keys, observed=True).ngroups
    out = _top_n_subset(df, top_n)
    if verbose:
        n_queries_after = out.groupby(keys, observed=True).ngroups
        print(f"Kept {n_queries_after:,} / {n_queries_before:,} query albums with >={top_n} recs")
    return out


def _norm_str(s) -> str:
    """Same normalization as ingestion/albums.ipynb (quotes, accents, whitespace)."""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):
        s = s.replace(ch, "'")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s)


def _norm(s: pd.Series) -> pd.Series:
    return s.fillna("").map(_norm_str)


def query_key(df):
    """Normalized artist::album key for query albums."""
    return _norm(df["query_artist"]) + "::" + _norm(df["query_album"])


def rec_key(df):
    """Normalized artist::album key for recommended albums."""
    return _norm(df["rec_artist"]) + "::" + _norm(df["rec_album"])


def metric_row(metric, recommender=None, **values):
    """One metrics row for pd.DataFrame append/concat."""
    row = {"metric": metric, **values}
    if recommender is not None:
        row["recommender"] = recommender
    return row


In [ ]:
# Load the album catalog from the CSV file containing metadata for all albums reviewed in the dataset
catalog = pd.read_csv(CATALOG_PATH)
catalog["key"] = _norm(catalog["artist"]) + "::" + _norm(catalog["album"])
catalog_reviews = catalog.set_index("key")["review_count"]
catalog_ids = catalog.set_index("key")["album_id"]
print(f"Catalog: {len(catalog):,} albums")

In [ ]:
# Load and sanitize recommendations from our recommender.
recs = pd.read_parquet(RECS_PATH)
recs = recs.drop(columns=["rec_length_flag"])

In [ ]:
# Load and sanitize recommendations from Last.fm benchmark recommender.
df = pd.read_csv(BENCH_PATH)
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")

# keep successful recommendations only
df = df[df["status"] == "ok"].reset_index(drop=True)

n_albums = df["album_id"].nunique()
print(f"File: {BENCH_PATH.name}")
print(f"Strategy: {df['strategy'].iloc[0]}  |  Albums: {n_albums:,}  |  Rec rows: {len(df):,}")

baseline = df.rename(columns={
    "artist": "query_artist",
    "album":  "query_album",
}).reindex(columns=[
    "album_id", "query_artist", "query_album", "rec_artist", "rec_album",
    "score", "rank", "seed_listeners", "rec_listeners",
]).copy()


baseline_n = keep_top_n(baseline)
baseline_1 = keep_top_n(baseline, 1)

### Match to baseline queries

Keep only embedding recs whose query album appears in the Last.fm baseline (normalized artist::album match). If the embedding corpus has duplicate spellings of the same album (e.g. curly vs straight apostrophes), keep one variant per normalized key so query counts align with the baseline.

In [ ]:
baseline_query_keys = set(query_key(baseline))

def filter_to_baseline(df):
    out = df.loc[query_key(df).isin(baseline_query_keys)].copy()
    out["_query_key"] = query_key(out)

    canonical = (
        out.groupby("_query_key", observed=True)[["query_artist", "query_album"]]
        .first()
        .rename(columns={"query_artist": "_canon_artist", "query_album": "_canon_album"})
    )
    out = out.join(canonical, on="_query_key")
    out = out[
        (out["query_artist"] == out["_canon_artist"])
        & (out["query_album"] == out["_canon_album"])
    ].drop(columns=["_query_key", "_canon_artist", "_canon_album"]).reset_index(drop=True)

    n_queries = out.groupby(["query_artist", "query_album"], observed=True).ngroups
    print(f"{len(out):,} rows, {n_queries:,} queries")
    return out

print("Embedding recs (matched to baseline):")
recs_matched = filter_to_baseline(recs)
recs_1 = keep_top_n(recs_matched, 1)
recs_n = keep_top_n(recs_matched)

# Comparisons

Each recommender has two views: `*_1` (rank-1 only — top-pick metrics) and `*_n` (full top-N lists — list-level metrics). Embedding recs are restricted to query albums present in the Last.fm baseline.

## Repetition

Do different query albums get steered toward the same recommended **albums** and **artists**? We sweep list depth `n = 1 … TOP_N` and measure how concentrated recommendations are.

The embedding model draws from a **much smaller catalog** than Last.fm, so some extra repetition is expected.

**How to read:** focus on **n = 1** (top pick) and **n = TOP_N** (full list). **Unique share** = distinct targets ÷ rec slots. **Hubs** = distinct targets recommended to more than one query. A widening emb vs base gap at full top-N means later ranks funnel into shared targets; artist repetition is usually the bigger concern. Treat Last.fm as the diversity ceiling.

### Albums

At each `n`:

- **Unique share** — fraction of rec slots that point to distinct album titles (higher = more diverse).
- **Repeated hubs** — count of distinct albums recommended as a top pick for *more than one* query album. Each hub is a target the recommender keeps returning across unrelated queries; a high count means many queries get steered toward the same albums.

In [ ]:
def repeated_stats(df, rec_col):
    counts = df[rec_col].str.strip().str.lower().value_counts()
    n_total = len(df)
    n_unique = counts.size
    n_repeated = int((counts > 1).sum())
    return {
        "n_total": n_total,
        "n_queries": df.groupby(["query_artist", "query_album"], observed=True).ngroups,
        "n_unique": n_unique,
        "n_repeated": n_repeated,
        "unique_share": n_unique / n_total if n_total else float("nan"),
        "repeated_share": n_repeated / n_unique if n_unique else float("nan"),
    }


def _top_n_subset(df, top_n):
    """Keep ranks 1..top_n; quiet helper (also defined in Data Prep)."""
    sub = df[df["rank"] <= top_n].copy()
    keys = ["query_artist", "query_album"]
    n_recs = sub.groupby(keys, observed=True)["rank"].transform("count")
    return sub[n_recs >= top_n].reset_index(drop=True)


def repeated_sweep(recs_df, baseline_df, rec_col, metric, top_n=TOP_N):
    rows = []
    for n in range(1, top_n + 1):
        for label, source in [("embedding recs", recs_df), ("last.fm baseline", baseline_df)]:
            sub = _top_n_subset(source, n)
            stats = repeated_stats(sub, rec_col)
            rows.append({"metric": metric, "n": n, "recommender": label, **stats})
    return pd.DataFrame(rows)


repeated_sweep_albums = repeated_sweep(recs_matched, baseline, "rec_album", "repeated_albums")
repeated_sweep_albums

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for label, color in [
    ("embedding recs", "C0"),
    ("last.fm baseline", "C1"),
]:
    sub = repeated_sweep_albums[repeated_sweep_albums["recommender"] == label]
    axes[0].plot(sub["n"], sub["unique_share"], marker="o", label=label, color=color)
    axes[1].plot(sub["n"], sub["n_repeated"], marker="o", label=label, color=color)

axes[0].set(xlabel="n (top ranks kept)", ylabel="Unique albums / total recs", title="Diversity")
axes[0].set_xticks(range(1, TOP_N + 1))
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].set(xlabel="n (top ranks kept)", ylabel="# unique albums recommended > once", title="Repeated hubs")
axes[1].set_xticks(range(1, TOP_N + 1))
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

### Artists

Same sweep on recommended **artist** names. Artist-level hub counts are often higher than album-level because one act can appear on many different albums.


In [ ]:
repeated_sweep_artists = repeated_sweep(recs_matched, baseline, "rec_artist", "repeated_artists")
repeated_sweep_artists

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for label, color in [
    ("embedding recs", "C0"),
    ("last.fm baseline", "C1"),
]:
    sub = repeated_sweep_artists[repeated_sweep_artists["recommender"] == label]
    axes[0].plot(sub["n"], sub["unique_share"], marker="o", label=label, color=color)
    axes[1].plot(sub["n"], sub["n_repeated"], marker="o", label=label, color=color)

axes[0].set(xlabel="n (top ranks kept)", ylabel="Unique artists / total recs", title="Diversity")
axes[0].set_xticks(range(1, TOP_N + 1))
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].set(xlabel="n (top ranks kept)", ylabel="# unique artists recommended > once", title="Repeated hubs")
axes[1].set_xticks(range(1, TOP_N + 1))
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


## Variety

How spread out is each query's top-N list? **Repetition** measures whether different queries hit the same targets globally; **variety** measures diversity *within* a single query's list.

- **Variety ratio** = distinct rec targets ÷ list length (1.0 = every slot differs).
- **All distinct** = share of queries whose full list has no repeats.
- **Single-target only** = share of queries where every rec is the same target.

**How to read:** compare emb vs base on full top-N lists. Lower variety ratio or all-distinct share = less spread within each list. Album variety often nears the ceiling; artist variety shows the larger gap. Low single-target share means repetition shows up globally, not within one list.

### Albums

Distinct recommended **albums** (artist::album key) within each query's top-N list.

In [ ]:
def list_variety_stats(df, kind):
    """Distinct rec targets within each query's recommendation list."""
    keys = ["query_artist", "query_album"]
    if kind == "album":
        target = rec_key(df)
    else:
        target = _norm(df["rec_artist"])

    stats = pd.DataFrame({"target": target, "n_recs": 1}).join(df[keys])
    stats = stats.groupby(keys, observed=True).agg(
        n_recs=("n_recs", "sum"),
        n_unique=("target", "nunique"),
    )
    stats["ratio"] = stats["n_unique"] / stats["n_recs"]
    return {
        "n_queries": len(stats),
        "mean_unique": stats["n_unique"].mean(),
        "mean_variety_ratio": stats["ratio"].mean(),
        "all_distinct_share": (stats["n_unique"] == stats["n_recs"]).mean(),
        "single_target_share": (stats["n_unique"] == 1).mean(),
    }


def _print_variety(row, label, top_n=TOP_N):
    print(f"[{row['recommender']}]")
    print(f"  Queries:              {row['n_queries']:,.0f}")
    print(f"  Mean unique {label}:  {row['mean_unique']:.2f} / {top_n}")
    print(f"  Mean variety ratio:   {row['mean_variety_ratio']:.1%}")
    print(f"  All distinct:         {row['all_distinct_share']:.1%}")
    print(f"  Single {label} only:  {row['single_target_share']:.1%}")
    print()


sources = [("embedding recs", recs_n), ("last.fm baseline", baseline_n)]
album_variety = pd.DataFrame(
    [metric_row("album_variety", label, **list_variety_stats(df, "album")) for label, df in sources]
)

for _, row in album_variety.iterrows():
    _print_variety(row, "albums")

album_variety

### Artists

Distinct recommended **artists** within each query's top-N list. Variety is often lower here because one act can fill multiple album slots.

In [ ]:
artist_variety = pd.DataFrame(
    [metric_row("artist_variety", label, **list_variety_stats(df, "artist")) for label, df in sources]
)

for _, row in artist_variety.iterrows():
    _print_variety(row, "artists")

artist_variety

## Reciprocity

If artist A recommends artist B, does B recommend A back?

Each recommendation is a directed **edge**; a reciprocal pair `(A→B, B→A)` means mutual pointing. High reciprocity suggests **closed neighborhoods**. We measure at **artist** and **album** level.

**How to read:** **Reciprocal rate** = share of edges with a reverse partner. Artist level is more informative — mutual act links are more common than exact album reversals. Higher emb rate = embedding builds tighter mutual neighborhoods.

### Albums

Album-to-album edges use `query_artist::query_album → rec_artist::rec_album`. **Reciprocal rate** = share of edges whose reverse partner also appears.

In [ ]:
def reciprocity_stats(edges):
    reciprocal = sum(1 for a, b in edges if (b, a) in edges)
    n_edges = len(edges)
    return {
        "n_edges": n_edges,
        "n_reciprocal": reciprocal,
        "reciprocal_rate": reciprocal / n_edges if n_edges else float("nan"),
    }


def album_reciprocity_stats(df):
    edges = set(zip(
        _norm(df["query_artist"]) + "::" + _norm(df["query_album"]),
        _norm(df["rec_artist"]) + "::" + _norm(df["rec_album"]),
    ))
    return reciprocity_stats(edges)


def _print_reciprocity(row):
    print(f"[{row['recommender']}]")
    print(f"  Edges:       {row['n_edges']:,.0f}")
    print(f"  Reciprocal:  {row['n_reciprocal']:,.0f}")
    print(f"  Rate:        {row['reciprocal_rate']:.1%}")
    print()


sources = [("embedding recs", recs_n), ("last.fm baseline", baseline_n)]
album_reciprocity = pd.DataFrame(
    [metric_row("album_reciprocity", label, **album_reciprocity_stats(df)) for label, df in sources]
)

for _, row in album_reciprocity.iterrows():
    _print_reciprocity(row)

album_reciprocity

### Artists

Artist-to-artist edges collapse to act names. Reciprocity is usually higher here than at album level.

In [ ]:
def artist_reciprocity_stats(df):
    edges = set(zip(_norm(df["query_artist"]), _norm(df["rec_artist"])))
    return reciprocity_stats(edges)


artist_reciprocity = pd.DataFrame(
    [metric_row("artist_reciprocity", label, **artist_reciprocity_stats(df)) for label, df in sources]
)

for _, row in artist_reciprocity.iterrows():
    _print_reciprocity(row)

artist_reciprocity

## Novelty

How uncommon are the recommended albums across query lists?

- **List reach** — avg # query lists each recommended album appears in (lower = more novel).
- **Singleton share** — share of rec slots where the album appears on only one list.
- **Normalized novelty** — self-information on 0–1 (1 = unique to one list).
- **Effective catalog** — distinct albums implied if recs were spread uniformly.

**How to read:** lower reach and higher singleton share / novelty / effective catalog = more novel. Metrics should move together; higher reach with lower singleton share means the same albums recur across queries (see Repetition hubs).

In [ ]:
def _novelty_base(df):
    n_lists = df.groupby(["query_artist", "query_album"], observed=True).ngroups
    key = rec_key(df)
    reach = key.map(key.value_counts())
    return n_lists, reach


def list_reach_stats(df):
    n_lists, reach = _novelty_base(df)
    return {"n_lists": n_lists, "mean_reach": reach.mean(), "median_reach": reach.median()}


def singleton_share_stats(df):
    _, reach = _novelty_base(df)
    return {"singleton_share": (reach == 1).mean()}


def normalized_novelty_stats(df):
    n_lists, reach = _novelty_base(df)
    log_n = np.log2(n_lists)
    norm = np.log2(n_lists / reach) / log_n if log_n else reach * 0
    return {"mean_novelty": norm.mean(), "median_novelty": norm.median()}


def effective_catalog_stats(df):
    key = rec_key(df)
    counts = key.value_counts()
    p = counts / counts.sum()
    return {"effective_catalog": 2 ** (-(p * np.log2(p)).sum())}


sources = [("embedding recs", recs_n), ("last.fm baseline", baseline_n)]
rows = []
for label, df in sources:
    row = {"recommender": label}
    row.update(list_reach_stats(df))
    row.update(singleton_share_stats(df))
    row.update(normalized_novelty_stats(df))
    row.update(effective_catalog_stats(df))
    rows.append(row)

novelty = pd.DataFrame(rows)

for _, row in novelty.iterrows():
    print(f"[{row['recommender']}]")
    print(f"  List reach (mean):   {row['mean_reach']:.2f}")
    print(f"  Singleton share:     {row['singleton_share']:.1%}")
    print(f"  Normalized novelty:  {row['mean_novelty']:.1%}")
    print(f"  Effective catalog:   {row['effective_catalog']:,.0f}")
    print()

novelty

## Catalog overlap & concentration

How do recommendations relate to the review catalog (`albums.csv`)? Embedding is index-constrained; Last.fm is open-world.

- **In-corpus rec share** — share of rec slots landing in `albums.csv`.
- **Distinct in-corpus albums** — review-catalog albums recommended at least once.
- **Review-catalog share** — index utilization (emb) vs in-corpus overlap (Last.fm).
- **Top-1% concentration** — share of slots going to the most-recommended albums.

**How to read:** in-corpus share and concentration compare directly. Review-catalog share is asymmetric — index coverage for embedding, overlap with your corpus for Last.fm. Low Last.fm in-corpus share reflects its open catalog, not "using less" of albums.csv.

In [ ]:
def catalog_overlap_stats(df):
    key = rec_key(df)
    in_catalog = key.isin(catalog_reviews.index)
    in_corpus = key[in_catalog]
    counts = key.value_counts()
    top1pct = max(1, int(len(counts) * 0.01))
    n_catalog = len(catalog)
    n_in_corpus = in_corpus.nunique()
    return {
        "in_corpus_rec_share": in_catalog.mean(),
        "n_distinct_recs": key.nunique(),
        "n_in_corpus_albums": n_in_corpus,
        "n_catalog_albums": n_catalog,
        "review_catalog_share": n_in_corpus / n_catalog if n_catalog else float("nan"),
        "top1pct_slot_share": counts.head(top1pct).sum() / counts.sum() if len(counts) else float("nan"),
    }


sources = [("embedding recs", recs_n), ("last.fm baseline", baseline_n)]
catalog_overlap = pd.DataFrame(
    [metric_row("catalog_overlap", label, **catalog_overlap_stats(df)) for label, df in sources]
)

for _, row in catalog_overlap.iterrows():
    label = "Index coverage" if row["recommender"] == "embedding recs" else "In-corpus overlap"
    print(f"[{row['recommender']}]")
    print(f"  In-corpus recs:       {row['in_corpus_rec_share']:.1%}")
    print(f"  Distinct rec albums:  {row['n_distinct_recs']:,.0f}  ({row['n_in_corpus_albums']:,.0f} in corpus)")
    print(f"  {label}:            {row['n_in_corpus_albums']:,.0f} / {row['n_catalog_albums']:,.0f} ({row['review_catalog_share']:.1%})")
    print(f"  Top 1% concentration: {row['top1pct_slot_share']:.1%}")
    print()

catalog_overlap

## Popularity bias

Do recommended albums skew more popular than the catalog — or, for Last.fm, than the seed albums?

- **Review count** — mean `review_count` of in-catalog recs vs catalog mean (both recommenders; `albums.csv` only). Ratio > 1 = popularity bias.
- **Last.fm listeners** — median listeners of rec vs seed album (baseline only; from batch CSV).

**How to read:** review-count stats only cover in-corpus recs — check catalog-overlap coverage first. Last.fm listener bias is a separate baseline-only signal (rec vs seed), not directly comparable to review count.

### Review count

Popularity proxy from the review catalog. **Ratio > 1** means in-catalog recommendations lean toward albums with more reviews than the catalog average.

In [ ]:
def review_count_bias_stats(df):
    rec_reviews = rec_key(df).map(catalog_reviews)
    in_corpus = rec_reviews.notna()
    rec_reviews = rec_reviews[in_corpus]
    catalog_mean = catalog_reviews.mean()
    return {
        "in_corpus_rec_share": in_corpus.mean(),
        "mean_reviews_recs": rec_reviews.mean(),
        "mean_reviews_catalog": catalog_mean,
        "mean_reviews_ratio": rec_reviews.mean() / catalog_mean,
        "recs_3plus_share": (rec_reviews >= 3).mean(),
        "catalog_3plus_share": (catalog_reviews >= 3).mean(),
    }


sources = [("embedding recs", recs_n), ("last.fm baseline", baseline_n)]
review_count_bias = pd.DataFrame(
    [metric_row("review_count_bias", label, **review_count_bias_stats(df)) for label, df in sources]
)

for _, row in review_count_bias.iterrows():
    print(f"[{row['recommender']}]")
    print(f"  In-corpus recs:     {row['in_corpus_rec_share']:.1%}")
    print(f"  Mean reviews ratio: {row['mean_reviews_ratio']:.2f}x  (catalog mean: {row['mean_reviews_catalog']:.2f})")
    print(f"  3+ reviews share:   {row['recs_3plus_share']:.1%}  (catalog: {row['catalog_3plus_share']:.1%})")
    print()

review_count_bias

### Last.fm listeners

Are Last.fm recommendations more listened-to than the query albums that produced them? Uses `rec_listeners` / `seed_listeners` from the batch CSV — baseline only.

In [ ]:
def listener_popularity_stats(df):
    sub = df.dropna(subset=["rec_listeners", "seed_listeners"])
    if sub.empty:
        return {
            "n_rows_with_data": 0,
            "median_rec_listeners": float("nan"),
            "median_seed_listeners": float("nan"),
            "median_rec_seed_ratio": float("nan"),
            "rec_more_popular_share": float("nan"),
        }
    ratio = (sub["rec_listeners"] + 1) / (sub["seed_listeners"] + 1)
    return {
        "n_rows_with_data": len(sub),
        "median_rec_listeners": sub["rec_listeners"].median(),
        "median_seed_listeners": sub["seed_listeners"].median(),
        "median_rec_seed_ratio": ratio.median(),
        "rec_more_popular_share": (ratio > 1).mean(),
    }


listener_popularity = pd.DataFrame(
    [metric_row("listener_popularity", "last.fm baseline", **listener_popularity_stats(baseline_n))]
)

row = listener_popularity.iloc[0]
if row["n_rows_with_data"] == 0:
    print("[last.fm baseline]  no listener data")
else:
    print(f"[{row['recommender']}]")
    print(f"  Rows with data:       {row['n_rows_with_data']:,.0f}")
    print(f"  Median rec listeners: {row['median_rec_listeners']:,.0f}")
    print(f"  Median seed listeners:{row['median_seed_listeners']:,.0f}")
    print(f"  Rec/seed ratio:       {row['median_rec_seed_ratio']:.2f}x")
    print(f"  Rec more popular:     {row['rec_more_popular_share']:.1%}")
    print()

listener_popularity

## Unexpectedness

How much do the two recommenders disagree on shared queries?

- **Full-list Jaccard** — overlap of complete top-N lists (0 = none, 1 = identical). Last.fm includes many out-of-corpus albums, so this mixes catalog mismatch with disagreement.
- **In-corpus Jaccard** — same measure on `albums.csv` recs only, for queries where **both** sides have ≥ TOP_N in-corpus albums.

**How to read:** low full-list Jaccard is expected when Last.fm sits outside your index. Use in-corpus Jaccard for agreement on the shared evaluation universe. Low in-corpus overlap still means the models disagree on picks, not just catalogs.

In [ ]:
def rec_lists(df):
    frame = pd.DataFrame({
        "query": _norm(df["query_artist"]) + "::" + _norm(df["query_album"]),
        "rec": rec_key(df),
    })
    return frame.groupby("query")["rec"].agg(set)


def _jaccard(lists_a, lists_b, queries):
    return pd.Series(
        [len(lists_a[q] & lists_b[q]) / len(lists_a[q] | lists_b[q]) for q in queries],
        index=queries,
    )


def unexpectedness_stats(df_a, df_b, min_in_corpus=TOP_N):
    lists_a, lists_b = rec_lists(df_a), rec_lists(df_b)
    shared = lists_a.index.intersection(lists_b.index)
    empty = {"n_shared_queries": 0, "min_in_corpus": min_in_corpus}
    nan_metrics = {
        "mean_jaccard": float("nan"),
        "zero_overlap_share": float("nan"),
        "any_overlap_share": float("nan"),
        "n_comparable_queries": 0,
        "comparable_query_share": float("nan"),
        "mean_jaccard_in_corpus": float("nan"),
        "zero_overlap_in_corpus_share": float("nan"),
        "any_overlap_in_corpus_share": float("nan"),
    }
    if shared.empty:
        return {**empty, **nan_metrics}

    j_full = _jaccard(lists_a, lists_b, shared)
    catalog_keys = set(catalog_reviews.index)
    in_a = lists_a.map(lambda s: s & catalog_keys)
    in_b = lists_b.map(lambda s: s & catalog_keys)
    comparable = [q for q in shared if len(in_a[q]) >= min_in_corpus and len(in_b[q]) >= min_in_corpus]
    out = {
        "n_shared_queries": len(shared),
        "min_in_corpus": min_in_corpus,
        "mean_jaccard": j_full.mean(),
        "zero_overlap_share": (j_full == 0).mean(),
        "any_overlap_share": (j_full > 0).mean(),
        "n_comparable_queries": len(comparable),
        "comparable_query_share": len(comparable) / len(shared),
    }
    if comparable:
        j_inc = _jaccard(in_a, in_b, comparable)
        out.update(
            {
                "mean_jaccard_in_corpus": j_inc.mean(),
                "zero_overlap_in_corpus_share": (j_inc == 0).mean(),
                "any_overlap_in_corpus_share": (j_inc > 0).mean(),
            }
        )
    else:
        out.update(
            {
                "mean_jaccard_in_corpus": float("nan"),
                "zero_overlap_in_corpus_share": float("nan"),
                "any_overlap_in_corpus_share": float("nan"),
            }
        )
    return out


unexpectedness = pd.DataFrame([metric_row("unexpectedness", **unexpectedness_stats(recs_n, baseline_n))])
row = unexpectedness.iloc[0]

if row["n_shared_queries"] == 0:
    print("No shared query albums")
else:
    print(f"Shared queries: {row['n_shared_queries']:,.0f}")
    print()
    print("Full-list Jaccard")
    print(f"  Mean:         {row['mean_jaccard']:.1%}")
    print(f"  Zero overlap: {row['zero_overlap_share']:.1%} of queries")
    print(f"  Any overlap:  {row['any_overlap_share']:.1%} of queries")
    print()
    print(f"In-corpus Jaccard (both sides have >={int(row['min_in_corpus'])} albums.csv recs)")
    print(f"  Comparable:   {row['n_comparable_queries']:,.0f} / {row['n_shared_queries']:,.0f} ({row['comparable_query_share']:.1%})")
    if row["n_comparable_queries"]:
        print(f"  Mean:         {row['mean_jaccard_in_corpus']:.1%}")
        print(f"  Zero overlap: {row['zero_overlap_in_corpus_share']:.1%} of comparable queries")
        print(f"  Any overlap:  {row['any_overlap_in_corpus_share']:.1%} of comparable queries")
    print()

unexpectedness

## Serendipity-lite

How often are recs "relevant but not obvious"?

This is a **rough proxy**, not full serendipity (which also needs relevance and user surprise). Each **rec slot** is scored independently.

- **Same-artist share** — recommended artist equals the query artist. Last.fm tries to exclude these; embedding is not filtered the same way.
- **Serendipitous (all slots)** — different artist, in `albums.csv`, and `review_count` ≤ catalog median (currently 1).
- **Serendipitous (in-corpus only)** — same rule, but denominator is only in-corpus slots.

**How to read:** compare **serendipitous (in-corpus)** for a fair emb vs Last.fm read. All-slots rates are dominated by in-corpus coverage (~96% emb vs ~31% Last.fm). Out-of-corpus recs never count as serendipitous. "Obscure" here is a very low bar (≤1 review in our corpus).

In [ ]:
def serendipity_lite_stats(df):
    same_artist = _norm(df["rec_artist"]) == _norm(df["query_artist"])
    rec_reviews = rec_key(df).map(catalog_reviews)
    median_reviews = catalog_reviews.median()
    in_corpus = rec_reviews.notna()
    obscure = rec_reviews <= median_reviews
    serendipitous = ~same_artist & obscure
    return {
        "catalog_median_review_count": median_reviews,
        "in_corpus_share": in_corpus.mean(),
        "same_artist_share": same_artist.mean(),
        "serendipitous_share": serendipitous.mean(),
        "serendipitous_in_corpus_share": serendipitous[in_corpus].mean() if in_corpus.any() else float("nan"),
    }


sources = [("embedding recs", recs_n), ("last.fm baseline", baseline_n)]
serendipity_lite = pd.DataFrame(
    [metric_row("serendipity_lite", label, **serendipity_lite_stats(df)) for label, df in sources]
)

for _, row in serendipity_lite.iterrows():
    print(f"[{row['recommender']}]")
    print(f"  In-corpus recs:              {row['in_corpus_share']:.1%}")
    print(f"  Same-artist recs:            {row['same_artist_share']:.1%}")
    print(f"  Serendipitous (all slots):   {row['serendipitous_share']:.1%}")
    print(f"  Serendipitous (in-corpus):   {row['serendipitous_in_corpus_share']:.1%}")
    print(f"  (median review_count = {row['catalog_median_review_count']:.0f})")
    print()

serendipity_lite